# SHAP and Fairness Analysis

This notebook reproduces global and local SHAP explanations and reviews region-based operational diagnostics. It does not claim protected-class fairness: the public Olist files contain no protected demographic attributes. SHAP attributes model-score contributions and is not causal evidence (Lundberg & Lee, 2017).

In [ ]:
from pathlib import Path
import joblib, pandas as pd, shap
from train import load_model_table, FEATURES, OUT, ART

## Rebuild the chronological hold-out

The later 20% by purchase timestamp is held out. The outcome is not passed to the explainer.

In [ ]:
data = load_model_table().sort_values('order_purchase_timestamp').reset_index(drop=True)
cut = int(len(data) * 0.8)
test = data.iloc[cut:]
X_test = test[FEATURES]
y_test = test['late']
model = joblib.load(ART / 'delivery_risk_model.joblib')
prob = model.predict_proba(X_test)[:, 1]
print(f'Hold-out rows: {len(test):,}; late rate: {y_test.mean():.3%}')

## Global explanation

A linear SHAP explainer is applied to the selected logistic model after its preprocessing pipeline. The bounded test sample keeps the exercise reproducible on a standard laptop.

In [ ]:
X_train = data.iloc[:cut][FEATURES]
X_transformed = model.named_steps['prep'].transform(X_test.iloc[:600])
background = model.named_steps['prep'].transform(X_train.iloc[:300])
explainer = shap.LinearExplainer(model.named_steps['model'], background)
shap_values = explainer(X_transformed)
feature_names = model.named_steps['prep'].get_feature_names_out()
shap.summary_plot(shap_values.values, X_transformed, feature_names=feature_names, max_display=15)

## Local explanation

The waterfall decomposes one held-out score. It explains the fitted model's behavior, not what would causally change delivery outcomes.

In [ ]:
local = explainer(X_transformed[:1])
row = X_transformed[0].toarray().ravel() if hasattr(X_transformed[0], 'toarray') else X_transformed[0]
shap.plots.waterfall(shap.Explanation(values=local.values[0], base_values=float(local.base_values.reshape(-1)[0]), data=row, feature_names=feature_names), max_display=12)

## Fairness diagnostics

The attached table compares five Brazilian macroregions as operational slices only. No protected attribute is available. Review group counts and errors as service-investigation signals, not demographic fairness proof. Do not apply threshold mitigation by region without governance approval.

In [ ]:
fairness = pd.read_csv(OUT / 'fairness_metrics.csv')
fairness.sort_values('predicted_positive_rate', ascending=False)